# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [ ]:
!gdown https://drive.google.com/uc?id=1Z4XXWwVnNZ0q18aw4UFh_ceA8zFxlkHz

Downloading...
From: https://drive.google.com/uc?id=1Z4XXWwVnNZ0q18aw4UFh_ceA8zFxlkHz
To: /content/netflix_catalogue.csv
100% 142k/142k [00:00<00:00, 79.7MB/s]


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [ ]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [ ]:
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# Filter to TV-14, TV-MA, PG-13, R, PG only
ratings_to_filter = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
df_filtered = df[df['rating'].isin(ratings_to_filter)]

# Group by rating and decade and count titles
heatmap_data = df_filtered.groupby(['rating', 'decade']).size().reset_index(name='count')

# Create a pivot table for the heatmap
pivot_table = heatmap_data.pivot(index='rating', columns='decade', values='count').fillna(0)

# Sort decades chronologically
sorted_decades = sorted(pivot_table.columns)
pivot_table = pivot_table[sorted_decades]

# Sort ratings by a meaningful order for visualization
rating_display_order = ['PG', 'PG-13', 'R', 'TV-14', 'TV-MA']
pivot_table = pivot_table.reindex(rating_display_order, axis=0)

# Find the maximum count and its location for the title annotation
max_count = pivot_table.max().max()
max_location = pivot_table.stack().idxmax()
max_rating = max_location[0]
max_decade = max_location[1]

fig = px.imshow(
    pivot_table,
    text_auto=True,
    color_continuous_scale='Blues',   # Sequential colour scale (Blues)
    aspect='auto', width=1100, height=700
)

fig.update_layout(
    title=f"Heatmap of Titles by Content Rating and Release Decade (Highest: {max_rating} in {max_decade} with {max_count:.0f} titles)",
    font=dict(family='Arial', size=12),
    coloraxis_showscale=False,         # Values in cells make colorbar redundant
    margin=dict(l=60, r=40, t=80, b=40),
    xaxis=dict(title='Release Decade', showgrid=False),
    yaxis=dict(title='Content Rating', showgrid=False)
)
fig.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [ ]:
df_movies = df[df['type'] == 'Movie']
adds = df_movies.groupby('added_year').size().reset_index(name='new_titles')
adds = adds[(adds['added_year'] >= 2015) & (adds['added_year'] <= 2022)]

cumulative = adds['new_titles'].sum()

# Find the year with the largest single addition
max_addition_year = adds.loc[adds['new_titles'].idxmax()]

trace = go.Waterfall(
    x=adds['added_year'].astype(str).tolist() + ['Total 2015-2022'],  # building x-ticks labels
    y=adds['new_titles'].tolist() + [cumulative],                     # building y-ticks labels
    measure=['relative']*len(adds) + ['total'],   # relative = bars stack; total = final sum
    connector=dict(line=dict(color='#AAAAAA', dash='dot')),
    increasing=dict(marker_color='#70AD47'),       # green for additions (positive)
    totals=dict(marker_color='#2E75B6'),           # blue for the total bar
    texttemplate='%{y:,}',                         # shows the y values as annotation
    textposition='outside'
)

my_data = [trace]
fig = go.Figure(data=my_data)

fig.update_layout(
    title=f"Netflix's Movie Library Growth (2015-2022): Largest Single Addition in {max_addition_year['added_year']} ({max_addition_year['new_titles']:,} titles)",
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    yaxis=dict(gridcolor='#EEEEEE', title='Titles Added'),
    xaxis=dict(title='Year', showgrid=False),
    margin=dict(l=60, r=40, t=80, b=40), # Increased top margin for title
    showlegend=False,
    height=700
)
fig.show()